# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa — Exploration with `mlcroissant`
This notebook provides an end-to-end step-by-step template for loading, exploring, and analyzing a Croissant-JSON-LD dataset using the `mlcroissant` library, adhering to best practices for reproducible and AI-ready research.

### Dataset Source
The dataset's Croissant schema is accessible at:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` and supporting libraries are installed. Restart the notebook kernel if just installed.
!pip install mlcroissant pandas matplotlib

## 1. Data Loading
Let's load metadata and records from the FAIR² Mental Health Survey dataset with `mlcroissant`. This will allow us to programmatically inspect schema objects (e.g., record sets, fields, and columns) and their `@id` fields for reliable downstream reference.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load metadata and initialize the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\n{metadata.description}")
print(f"\nVersion: {metadata.version}\nLicense: {metadata.license}")
print(f"Date published: {metadata.datePublished}\nCite as: {metadata.citeAs}")

## 2. Data Overview
Let's inspect available record sets, their `@id`s, fields, and columns in this dataset.

**Tip:** All references will use `@id` fields for reproducibility and correct cross-referencing as described in the Croissant standard.

In [ ]:
# List all record sets and their @ids
record_sets = list(dataset.record_sets())
print(f"Found {len(record_sets)} record sets:\n")
for rs in record_sets:
    print(f"- Record set name: {rs.name}")
    print(f"  @id: {rs.id}")
    # List columns (fields) for this record set
    columns = rs.columns
    print("  Fields (@id: name):")
    for col in columns:
        print(f"    {col.id}: {col.name}")
    print()

For demonstration, let's examine a small sample of records (rows) from each record set, referencing by exact `@id`.

In [ ]:
# Print a preview (first few records) from each record set using its @id
N = 2  # number of samples to show
for rs in record_sets:
    print("="*60)
    print(f"Sample records from record set '{rs.name}' (@id: {rs.id}):")
    for idx, record in enumerate(dataset.records(record_set=rs.id)):
        print(f"Record {idx+1}: {record}")
        if idx >= N-1:
            break


## 3. Data Extraction
Now we'll load the main survey data into a DataFrame for analysis. We'll specify the relevant `@id` for the main record set and its fields. The main survey record set in this dataset is typically named something like `kilifi_survey` or similar (please confirm below).

For demonstration, we'll extract **all record sets** and create a pandas DataFrame for each, storing them in a dictionary keyed by their `@id`.

In [ ]:
# Extract data for all record sets
dataframes = {}
for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"Loaded record set '@id': {rs.id} ({len(df)} records, {len(df.columns)} columns)")

# Pick the main record set for Exploratory Data Analysis (e.g., main survey data)
main_record_set = record_sets[0].id if record_sets else None
if main_record_set:
    print(f"Available fields (@id) in main record set '{main_record_set}':\n", list(dataframes[main_record_set].columns))
    display(dataframes[main_record_set].head())


## 4. Exploratory Data Analysis (EDA)
We'll now perform basic analysis on the main survey DataFrame. This includes:
- Filtering records based on a numeric field (e.g., GAD-7 score)
- Normalizing the selected field
- Grouping by a key categorical variable (e.g., 'gender')

**Use field `@id`s in all operations.**

Let's find likely candidates for a numeric field and a group field:

In [ ]:
# Suggest likely numeric and group fields by name and column headers
df = dataframes[main_record_set]
print("Column names available in the main survey record set:")
for c in df.columns:
    print(f"- {c}")
# Example guess: gad7_total (GAD-7 survey), phq9_total, psq_score, age, etc.
# Example group: gender, marital_status, village, education, etc.

**Select field `@id`s for EDA**. Please replace these with the exact `@id` string as shown in previous steps if they differ. We'll use as example:
- Numeric field: `'gad7_total'` (Generalized Anxiety Disorder score)
- Group field: `'gender'`

---

In [ ]:
numeric_field_id = 'gad7_total'   # Use the actual @id from the schema
group_field_id = 'gender'         # Use the actual @id from the schema

threshold = 10
df = dataframes[main_record_set]
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} (GAD-7 score) > {threshold}:")
    display(filtered_df.head())

    # Normalize
    norm_col = numeric_field_id + '_normalized'
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by group_field_id
    if group_field_id in filtered_df.columns:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean {numeric_field_id} grouped by '{group_field_id}':")
        print(grouped.head())
    else:
        print(f"Column {group_field_id} not found in DataFrame columns.")
else:
    print(f"Column {numeric_field_id} not found in DataFrame columns.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field and its relationship by gender.

- **Histogram** of GAD-7 scores
- **Box plot** of GAD-7 by gender

Feel free to adapt the field `@id`s as appropriate from your schema.

In [ ]:
plt.figure(figsize=(10, 4))
# Histogram
if numeric_field_id in df.columns:
    df[numeric_field_id].hist(bins=15, alpha=0.7, color='teal')
    plt.xlabel('GAD-7 Total Score (@id: gad7_total)')
    plt.ylabel('Count')
    plt.title('Distribution of GAD-7 Scores')
    plt.show()

# Boxplot by group
if numeric_field_id in df.columns and group_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    df.boxplot(column=numeric_field_id, by=group_field_id, grid=False)
    plt.title(f'GAD-7 Score by {group_field_id}')
    plt.suptitle('')
    plt.xlabel(group_field_id)
    plt.ylabel('GAD-7 Score')
    plt.show()

## 6. Conclusion
In this notebook, we've shown how to:
- Load a Croissant-metadata dataset using `mlcroissant`
- Programmatically discover available record sets, fields, and their `@id`s
- Structure downstream analysis so all entities use their stable `@id` as reference
- Perform basic data cleaning and exploratory analysis, including normalization and grouping
- Visualize important distributions

For full documentation and advanced usage, see [`mlcroissant.readthedocs.io`](https://mlcroissant.readthedocs.io/) and your dataset's schema.📚

**Happy FAIR data science!**